In [5]:
import os, sys, pickle, torch, numpy as np, dgl
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import ListedColormap

# ========= Imports de ton repo =========
project_path = os.path.abspath(os.path.join(os.getcwd(), '..', ''))
if project_path not in sys.path:
    sys.path.append(project_path)

from hydra.utils import to_absolute_path
from python.create_dgl_dataset import TelemacDataset
from python.CustomMeshGraphNet import MeshGraphNet
from modulus.launch.utils import load_checkpoint

# ===== (OPTIONNEL) pour positions de nœuds, nécessaire pour les plots / GIF =====
MESH_SLF = "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Mesh8_corrige.slf"
try:
    from python.python_code.data_manip.extraction.telemac_file import TelemacFile
    from python.create_dgl_dataset import add_mesh_info
    _res_mesh = TelemacFile(MESH_SLF)
    POS, _ = add_mesh_info(_res_mesh)  # (N,2) numpy
    HAVE_POS = True
except Exception as e:
    print(f"[VISU] Pas de positions (MESH_SLF='{MESH_SLF}'). Raison: {e}")
    POS, HAVE_POS = None, False

# ===== config =====
DATA_BASE = "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Mesh8_base.bin"
PCKLS     = [
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_1_peak_2600_Group_1_peak_2600_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_2_peak_1000_Group_2_peak_1000_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_2_peak_1200_Group_2_peak_1200_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_2_peak_1600_Group_2_peak_1600_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_4_peak_2000_Group_4_peak_2000_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_1_peak_1200_Group_1_peak_1200_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_1_peak_2400_Group_1_peak_2400_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_3_peak_3400_Group_3_peak_3400_0_0-80_interpolated.pkl",
]
CKPT_DIR  = "/work/m24046/m24046mrcr/testAter/Configx8ShortPushFront"
CKPT_EPOCH= 900   # modèle pushforward+front

# Paramètres modèle (en ligne avec ta conf)
NUM_IN  = 9         # 6 statiques (4 one-hot + strickler + z) + 3 dyn (h,u,v)
NUM_E   = 3
NUM_OUT = 3
MP_LAYERS = 10
DO_CONCAT = True
SEGMENTS  = 0

# Indices des features dynamiques dans ndata['x']
DYN_START = 6   # [onehot(4), strickler(1), z(1)] -> 6 colonnes statiques
DYN_LEN   = 3   # (h,u,v)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===== helpers =====
def denorm(xn, mean, std):
    return xn * std + mean

def renorm(x, mean, std):
    return (x - mean) / (std + 1e-12)

# ====== Prépare dataset (pour récupérer base_graph + stats) ======
ds = TelemacDataset(
    name="telemac_test",
    data_dir=to_absolute_path(DATA_BASE),
    dynamic_data_files=[to_absolute_path(p) for p in PCKLS],  # peu importe ici
    split="test",
    ckpt_path=CKPT_DIR,
    normalize=True,           # important: on charge les stats depuis CKPT_DIR
    sequence_length=1,
    overlap=0,
)

base_graph = ds.base_graph.to(device)
node_stats = ds.node_stats

# ==== récupère static + stats depuis ton dataset ====
g_base = base_graph.to(device)
static_feat = g_base.ndata['static']              # (N,6)
DYN_START   = static_feat.shape[1]                # = 6 (sécurisé)
DYN_LEN     = 3

# Stats pour (h,u,v) et (delta_h,delta_u,delta_v)
mx = np.array([node_stats['h'].item(),        node_stats['u'].item(),        node_stats['v'].item()],        dtype=np.float32)
sx = np.array([node_stats['h_std'].item(),    node_stats['u_std'].item(),    node_stats['v_std'].item()],    dtype=np.float32)
my = np.array([node_stats['delta_h'].item(),  node_stats['delta_u'].item(),  node_stats['delta_v'].item()],  dtype=np.float32)
sy = np.array([node_stats['delta_h_std'].item(), node_stats['delta_u_std'].item(), node_stats['delta_v_std'].item()], dtype=np.float32)

# ====== Modèle + checkpoint ======
model = MeshGraphNet(
    NUM_IN, NUM_E, NUM_OUT,
    processor_size=MP_LAYERS,
    hidden_dim_processor=64,
    hidden_dim_node_encoder=64,
    hidden_dim_edge_encoder=64,
    hidden_dim_node_decoder=64,
    do_concat_trick=DO_CONCAT,
    num_processor_checkpoint_segments=SEGMENTS,
).to(device)
model.eval()

_ = load_checkpoint(
    to_absolute_path(CKPT_DIR),
    models=model,
    device=device,
    epoch=CKPT_EPOCH,
)

# ==== Récupération h_pred / h_gt sur une crue ====
def rollout_h_one_file(pkl_path, model, apply_bc=True):
    """
    Retourne:
      H_pred: (T, N)  h prédite à chaque pas (inclut t=0 = état initial)
      H_gt  : (T, N)  h TELEMACH à chaque pas
    """
    with open(pkl_path, 'rb') as f:
        dynamic_data = pickle.load(f)     # liste de (x_dyn, y) par pas
    T = len(dynamic_data); assert T >= 2

    # graph de travail + x(t=0)
    g = g_base.clone().to(device)
    x_dyn0 = dynamic_data[0][0].astype(np.float32)          # (N,3) h,u,v @ t=0 (non norm.)
    xn0    = renorm(x_dyn0, mx, sx)                         # normalisé
    g.ndata['x'] = torch.cat([static_feat, torch.from_numpy(xn0).to(device)], dim=1)

    # masques CL (depuis one-hot dans les 4 premières colonnes)
    onehot = g.ndata['x'][:, :4]
    q_mask = (onehot == torch.tensor([0,0,1,0], device=device)).all(dim=1)   # Prescribed Q
    h_mask = (onehot == torch.tensor([0,1,0,0], device=device)).all(dim=1)   # Prescribed H
    q_mask_np, h_mask_np = q_mask.detach().cpu().numpy(), h_mask.detach().cpu().numpy()

    # listes pour stocker h
    H_pred = []
    H_gt   = []

    # t=0 : état initial, préd=gt (on démarre de la vérité)
    H_pred.append(x_dyn0[:, 0].copy())
    H_gt.append(x_dyn0[:, 0].copy())

    for t in range(T-1):
        with torch.no_grad():
            y_pred_n = model(g.ndata['x'], g.edata['x'], g)   # (N,3) deltas norm.
        y_pred   = denorm(y_pred_n.detach().cpu().numpy(), my, sy)  # (N,3) deltas non norm.

        # état courant x_t (non norm.)
        xn_t = g.ndata['x'][:, DYN_START:DYN_START+DYN_LEN].detach().cpu().numpy()
        x_t  = denorm(xn_t, mx, sx)                                      # (N,3)

        # prédiction t+1
        x_t1_pred = x_t + y_pred                                         # (N,3)
        x_t1_gt   = dynamic_data[t+1][0].astype(np.float32)              # TELEMACH @ t+1

        if apply_bc:
            x_t1_pred[q_mask_np, :]   = x_t1_gt[q_mask_np, :]
            x_t1_pred[h_mask_np, 0:1] = x_t1_gt[h_mask_np, 0:1]

        # stocke h
        H_pred.append(x_t1_pred[:, 0].copy())
        H_gt.append(x_t1_gt[:, 0].copy())

        # réinjection pour le pas suivant
        xn_t1_pred = renorm(x_t1_pred, mx, sx)
        g.ndata['x'] = torch.cat([g.ndata['x'][:, :DYN_START],
                                  torch.from_numpy(xn_t1_pred).to(device)], dim=1)

    H_pred = np.stack(H_pred, axis=0)   # (T, N)
    H_gt   = np.stack(H_gt,   axis=0)   # (T, N)
    return H_pred, H_gt

# ==== Création du GIF side-by-side avec PillowWriter ====
def make_side_by_side_gif(pos, H_pred, H_gt, out_gif="side_by_side_binary.gif",
                          thr=0.01, fps=5):
    """
    Plot binaire :
      - bleu si h > thr
      - blanc sinon
    """
    T, N = H_pred.shape
    x, y = pos[:, 0], pos[:, 1]

    # Colormap binaire : 0 -> blanc, 1 -> bleu
    bin_cmap = ListedColormap(["white", "blue"])

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
    ax_pred, ax_gt = axes

    # Frame 0 : valeurs binaires
    hp0 = (H_pred[0] > thr).astype(int)
    hg0 = (H_gt[0]   > thr).astype(int)

    sc_pred = ax_pred.scatter(x, y, c=hp0, s=2, vmin=0, vmax=1, cmap=bin_cmap)
    sc_gt   = ax_gt.scatter(x, y, c=hg0, s=2, vmin=0, vmax=1, cmap=bin_cmap)

    ax_pred.set_title(f"h prédit (>{thr:.2f} m)")
    ax_gt.set_title(f"h TELEMACH (>{thr:.2f} m)")
    for ax in axes:
        ax.set_aspect("equal")
        ax.set_xlabel("x")
        ax.set_facecolor("white")
    ax_pred.set_ylabel("y")

    def update(frame):
        hp = (H_pred[frame] > thr).astype(int)
        hg = (H_gt[frame]   > thr).astype(int)
        sc_pred.set_array(hp)
        sc_gt.set_array(hg)
        fig.suptitle(f"t = {frame}")
        return sc_pred, sc_gt

    ani = animation.FuncAnimation(
        fig, update, frames=T, interval=1000.0/fps, blit=False
    )

    writer = animation.PillowWriter(fps=fps)
    ani.save(out_gif, writer=writer)
    plt.close(fig)
    print(f"GIF sauvegardé dans {out_gif}")
    
# ==== main simple ====
if __name__ == "__main__":
    if not HAVE_POS:
        raise RuntimeError("Impossible de faire la vidéo sans POS (coordonnées nœuds).")

    # On prend la première crue de la liste
    pkl_path = PCKLS[0]
    base_name = os.path.splitext(os.path.basename(pkl_path))[0]
    print(f"Déroulement sur : {pkl_path}")

    H_pred, H_gt = rollout_h_one_file(pkl_path, model, apply_bc=True)

    out_gif = f"{base_name}_side_by_side.gif"
    make_side_by_side_gif(POS, H_pred, H_gt, out_gif=out_gif, fps=4)



Loading normalization statistics...


[17:30:46 - checkpoint - INFO] Loaded model state dictionary /work/m24046/m24046mrcr/testAter/Configx8ShortPushFront/MeshGraphNet.0.900.mdlus to device cuda
[17:30:46 - checkpoint - INFO] Loaded checkpoint file /work/m24046/m24046mrcr/testAter/Configx8ShortPushFront/checkpoint.0.900.pt to device cuda


Déroulement sur : /work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_1_peak_2600_Group_1_peak_2600_0_0-80_interpolated.pkl


INFO:matplotlib.animation:Animation.save using <class 'matplotlib.animation.PillowWriter'>


GIF sauvegardé dans Group_1_peak_2600_Group_1_peak_2600_0_0-80_interpolated_side_by_side.gif
